In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from aquarel import load_theme

theme = load_theme("minimal_light")
theme.apply()
theme.apply_transforms()
import matplotlib as mpl
mpl.rcParams['axes.spines.left'] = True
mpl.rcParams['axes.spines.right'] = False
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.bottom'] = True
mpl.rcParams['axes.edgecolor'] = 'grey'

In [ ]:
FIGURE_DATA = Path("../figure_data")
traj = pd.read_csv(FIGURE_DATA / 'stability_trajectory.csv')

In [ ]:
acr_methods = sorted(traj['method'].unique())
acr_color_map = matplotlib.colormaps.get_cmap('tab10')
acr_colors = {m: acr_color_map(i) for i, m in enumerate(acr_methods)}

signal_metrics = ['d_statistic', 'mpd', 'parsimony']
signal_labels  = ['D-statistic', 'MPD', 'Parsimony score']
stability_metrics = ['gains', 'losses']
N_BINS = 5

In [ ]:
def plot_stability_panel(traj, stability_metric, signal_metric, signal_label, ax):
    """Plot mean ± SE of stability_metric binned by signal_metric, one line per method."""
    rows = []
    for method, grp in traj.groupby('method'):
        grp = grp.dropna(subset=[signal_metric, stability_metric])
        if len(grp) < N_BINS:
            continue
        grp = grp.copy()
        try:
            grp['bin'] = pd.qcut(grp[signal_metric], q=N_BINS, duplicates='drop')
        except ValueError:
            grp['bin'] = pd.cut(grp[signal_metric], bins=N_BINS)
        for b, bgrp in grp.groupby('bin', observed=True):
            rows.append({
                'method': method,
                'bin_mid': bgrp[signal_metric].mean(),
                'mean': bgrp[stability_metric].mean(),
                'se':   bgrp[stability_metric].sem(),
            })
    df_bins = pd.DataFrame(rows)
    for method, mdf in df_bins.groupby('method'):
        mdf = mdf.sort_values('bin_mid')
        ax.errorbar(mdf['bin_mid'], mdf['mean'], yerr=mdf['se'],
                    label=method, color=acr_colors[method],
                    marker='o', markersize=4, capsize=3, alpha=0.8)
    ax.set_xlabel(signal_label)
    ax.set_ylabel(stability_metric.capitalize())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for col, (sig, sig_lbl) in enumerate(zip(signal_metrics, signal_labels)):
    for row, stab in enumerate(stability_metrics):
        plot_stability_panel(traj, stab, sig, sig_lbl, axes[row, col])
plt.tight_layout()
plt.savefig('fig.svg', format='svg')
plt.show()

In [ ]:
fig_leg, ax_leg = plt.subplots(figsize=(3, len(acr_methods) * 0.4 + 0.5))
ax_leg.axis('off')
handles = [plt.Line2D([0], [0], color=acr_colors[m], lw=2, marker='o', markersize=5) for m in acr_methods]
ax_leg.legend(handles, acr_methods, loc='center', frameon=False)
plt.tight_layout()
plt.savefig('figleg.svg', format='svg')
plt.show()